In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

sns.set(style='whitegrid')

# Q0: Objective
# This project evaluates major technology stocks by analysing returns, volatility, and correlations
# to understand risk-return trade-offs and diversification potential.

In [ ]:
# Q1: Load datasets
company_df = pd.read_csv('big_tech_companies.csv')
stock_df = pd.read_csv('big_tech_stock_prices.csv')

display(company_df.head())
display(stock_df.head())

In [ ]:
# Q2: Dataset structure
print(company_df.shape, stock_df.shape)
print('Company columns:', company_df.columns.tolist())
print('Stock columns:', stock_df.columns.tolist())

In [ ]:
# Q3: Convert date and sort
# Date format is DD-MM-YYYY, so dayfirst=True is required
stock_df['date'] = pd.to_datetime(stock_df['date'], dayfirst=True)
stock_df = stock_df.sort_values(by=['stock_symbol', 'date'])

In [ ]:
# Q4: Missing values
print('Stock missing values:')
print(stock_df.isnull().sum())
print('\nCompany missing values:')
print(company_df.isnull().sum())

In [ ]:
# Q5: Remove missing values
stock_df = stock_df.dropna()
company_df = company_df.dropna()

In [ ]:
# Q6: Merge datasets on stock_symbol (the common key in both files)
df = pd.merge(stock_df, company_df, on='stock_symbol')
display(df.head())

In [ ]:
# Q7: Feature Engineering
df['Return'] = df.groupby('company')['close'].pct_change()
df['Month'] = df['date'].dt.month
df = df.dropna()

In [ ]:
# Q8: Summary statistics
print(df.groupby('company')['close'].describe())

In [ ]:
# Q9: Volume analysis
print(df.groupby('company')['volume'].mean())

In [ ]:
# Q10: Price Trend
plt.figure(figsize=(14, 6))
for company in df['company'].unique():
    subset = df[df['company'] == company]
    plt.plot(subset['date'], subset['close'], label=company)
plt.title('Stock Price Trends')
plt.xlabel('Date')
plt.ylabel('Closing Price (USD)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()
# Insight: Shows overall growth patterns and fluctuations across companies.

In [ ]:
# Q11: Average Price
avg_price = df.groupby('company')['close'].mean().sort_values(ascending=False)
avg_price.plot(kind='bar', figsize=(12, 5), color='steelblue')
plt.title('Average Closing Price by Company')
plt.xlabel('Company')
plt.ylabel('Average Close (USD)')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
# Insight: Higher average price reflects stronger market valuation.

In [ ]:
# Q12: Returns Distribution
plt.figure(figsize=(12, 6))
for company in df['company'].unique():
    sns.kdeplot(df[df['company'] == company]['Return'], label=company)
plt.title('Returns Distribution by Company')
plt.xlabel('Daily Return')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()
# Insight: Wider distribution means higher volatility (risk).

In [ ]:
# Q13: Volume vs Price
plt.figure(figsize=(10, 6))
sns.scatterplot(data=df, x='volume', y='close', hue='company', alpha=0.3, s=10)
plt.title('Volume vs Closing Price')
plt.xlabel('Volume')
plt.ylabel('Close Price (USD)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()
# Insight: Weak relationship suggests price is influenced by multiple factors.

In [ ]:
# Q14: Volatility Box Plot
plt.figure(figsize=(14, 6))
sns.boxplot(data=df, x='company', y='Return')
plt.title('Return Distribution (Volatility) by Company')
plt.xlabel('Company')
plt.ylabel('Daily Return')
plt.xticks(rotation=45, ha='right')
plt.tight_layout()
plt.show()
# Insight: Larger spread = higher risk.

In [ ]:
# Q15: Correlation Heatmap
pivot_df = df.pivot_table(index='date', columns='company', values='Return', aggfunc='mean')
plt.figure(figsize=(14, 10))
sns.heatmap(pivot_df.corr(), annot=True, fmt='.2f', cmap='coolwarm')
plt.title('Correlation Heatmap of Daily Returns')
plt.tight_layout()
plt.show()
# Insight: High correlation means limited diversification in tech sector.

In [ ]:
# Q16: Risk vs Return
mean_returns = df.groupby('company')['Return'].mean()
volatility = df.groupby('company')['Return'].std()

plt.figure(figsize=(10, 7))
plt.scatter(volatility, mean_returns, color='tomato', s=80, zorder=5)

for company in mean_returns.index:
    plt.annotate(
        company,
        (volatility[company], mean_returns[company]),
        textcoords='offset points',
        xytext=(6, 4),
        fontsize=8
    )

plt.xlabel('Risk (Volatility / Std Dev of Returns)')
plt.ylabel('Average Daily Return')
plt.title('Risk vs Return Trade-off')
plt.tight_layout()
plt.show()
# Insight: Top-right = high risk, high return. Bottom-left = low risk, low return.

In [ ]:
# Q17: Rolling Average — plotted for every company
df['MA_30'] = df.groupby('company')['close'].transform(lambda x: x.rolling(30).mean())

for company in sorted(df['company'].unique()):
    subset = df[df['company'] == company]

    plt.figure(figsize=(12, 5))
    plt.plot(subset['date'], subset['close'], label='Actual', alpha=0.6)
    plt.plot(subset['date'], subset['MA_30'], label='30-Day MA', linewidth=2)
    plt.title(f'Rolling 30-Day Moving Average - {company}')
    plt.xlabel('Date')
    plt.ylabel('Close Price (USD)')
    plt.legend()
    plt.tight_layout()
    plt.show()

# Insight: Moving average smooths noise and highlights long-term trends for each company.

In [ ]:
# Q18: Cumulative Return
df['Cumulative_Return'] = (1 + df['Return']).groupby(df['company']).cumprod()

plt.figure(figsize=(14, 6))
for company in df['company'].unique():
    subset = df[df['company'] == company]
    plt.plot(subset['date'], subset['Cumulative_Return'], label=company)
plt.title('Cumulative Returns Over Time')
plt.xlabel('Date')
plt.ylabel('Cumulative Return (1 = starting value)')
plt.legend(bbox_to_anchor=(1.05, 1), loc='upper left', fontsize=8)
plt.tight_layout()
plt.show()
# Insight: Reveals which stock performed best over the full period.

In [ ]:
# Q19: Returns and Risk
avg_returns = df.groupby('company')['Return'].mean()
volatility = df.groupby('company')['Return'].std()

print('Average Daily Returns:')
print(avg_returns.sort_values(ascending=False))
print('\nVolatility (Std Dev):')
print(volatility.sort_values(ascending=False))

In [ ]:
# Q20: Best performing stock (long-term cumulative return)
best_stock = df.groupby('company')['Cumulative_Return'].last().idxmax()
best_value = df.groupby('company')['Cumulative_Return'].last().max()
print(f'Best Long-Term Performer: {best_stock} (Cumulative Return: {best_value:.2f}x)')

In [ ]:
# MACHINE LEARNING

# Q21-Q23: Linear Regression Model, Predictions, and Evaluation for every company
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, r2_score

results = []

for company in sorted(df['company'].unique()):
    company_data = df[df['company'] == company].copy()
    company_data['Date_Ordinal'] = company_data['date'].map(pd.Timestamp.toordinal)

    X = company_data[['Date_Ordinal']]
    y = company_data['close']

    X_train, X_test, y_train, y_test = train_test_split(X, y, shuffle=False, test_size=0.25)

    model = LinearRegression()
    model.fit(X_train, y_train)
    predictions = model.predict(X_test)

    # Q22: Plot predictions for this company
    plt.figure(figsize=(12, 5))
    plt.plot(company_data['date'], company_data['close'], label='Actual', alpha=0.7)
    plt.plot(company_data['date'].iloc[-len(predictions):], predictions,
             label='Predicted (Linear)', linewidth=2, linestyle='--')
    plt.title(f'Linear Regression Prediction - {company}')
    plt.xlabel('Date')
    plt.ylabel('Close Price (USD)')
    plt.legend()
    plt.tight_layout()
    plt.show()

    # Q23: Model Evaluation
    mse = mean_squared_error(y_test, predictions)
    r2 = r2_score(y_test, predictions)
    results.append({'Company': company, 'MSE': round(mse, 4), 'R2': round(r2, 4)})
    print(f'{company:45s}  MSE: {mse:10.4f}  R2: {r2:.4f}')

print('\nModel Evaluation Summary:')
print(pd.DataFrame(results).to_string(index=False))

# Insight: Model captures general trend but fails on volatility - limitation of linear models.

In [ ]:
# FINAL INSIGHTS
print("""
- High-return stocks also show significantly higher volatility (risk-return trade-off)
- Strong correlation suggests limited diversification within the tech sector
- Stable stocks are better suited for conservative investors
- Cumulative returns reveal long-term winners
- Linear regression captures trend but not volatility
""")

# CONCLUSION
print("""
This analysis evaluates stock performance using returns, volatility, and correlation.
It demonstrates trade-offs between risk and return and highlights limitations of simple models.

This analysis can support investment decision-making by identifying stocks
based on risk tolerance and expected return.
""")